# Qdrant × Future AGI: Fixing an Agentic RAG System That Decayed as It Grew

The agent is broken on purpose and running against a live Qdrant collection. Each section reads the Future AGI dashboard to locate the failing layer, fixes that layer in Qdrant, and verifies the metric moves.

- **Chat UI** (`app.py`): experience the failure
- **This notebook**: perform the fix; the chat UI picks it up on the next question
- **Future AGI dashboard**: prove it worked

This notebook prints instant gold-label retrieval checks (recall@5, duplicate rate). Judged quality scores live on the Future AGI dashboard, never here.

## 0 · Setup

In [ ]:
import os, json
from pathlib import Path
from dotenv import load_dotenv
from qdrant_client import QdrantClient, models

import agent
from helpers import config, embeddings
from helpers.embeddings import dense, sparse, colbert

load_dotenv()
client = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ["QDRANT_API_KEY"])
COLLECTION = config.COLLECTION

# Start every run from the broken baseline: small dense model, no filter.
agent.set_retrieval(mode="minilm", current_only=False)
embeddings.warmup()   # load all models now, never during a live question
client.count(COLLECTION)

### Tracing (Future AGI): Rishav's Segment
Three lines in `agent.py` (the file on camera) instrument everything:

```python
trace_provider = register(project_type=ProjectType.OBSERVE, project_name="pokedex-rag")
LangChainInstrumentor().instrument(tracer_provider=trace_provider)
```

traceAI auto-instruments the LangGraph agent, so every process that runs it exports traces: this notebook, the Streamlit app, and the rehearsal scripts. Qdrant searches appear as retriever spans with no manual span code.

## 1 · The Pain: It Decayed as It Grew
The same golden queries, scored against the collection as the Pokédex actually grew: Gen 1 only (1.3k points) → through Gen 4 (8.4k) → the full dex (22.9k). Generated offline by `scaling_curve.py`; each stage contains exactly the documents that existed in that generation's era. Real growth, not a fake timeline: recall@5 falls from 0.67 to 0.39 as ten generations of lookalike species pile into the collection, and the duplicate rate was high from day one — the flaws were always there, growth is what made them hurt.

One thing to keep straight as the numbers move: this curve runs all 33 scored queries through the broken pipeline. Each fix section then zooms into its own slice — the queries mined to expose that specific flaw — so the per-fix baselines (0.64, 0.83) differ from the curve's 0.39. Same golden set, different lens.

In [ ]:
from IPython.display import Image
Image("data/scaling_curve.png")   # generated offline by scaling_curve.py

## 2 · Fix #1: Duplicate and Fragmented Chunks → Dedup the Collection
**Dashboard read:** groundedness and context relevance stay green, but the retrieval panel shows the same `doc_id` filling slot after slot of the top-k. That pattern is duplicates, not a weak model.

In [ ]:
# Duplicate rate in the top-k before the fix: a slot is wasted if it repeats a fragment
# with the same doc_id and text we already retrieved.
def dup_rate(query):
    chunks = agent.retrieve(query, mode="minilm")
    keys = [(c["doc_id"], c["text"]) for c in chunks]
    return keys, 1 - len(set(keys)) / len(keys)

keys, rate = dup_rate("Tell me the Pokedex entry for Gengar")
print(f"duplicate rate: {rate:.0%}")
for k in keys: print(" ", k[0])

In [ ]:
# THE FIX: delete duplicate points with the same doc_id + chunk_index, and keep one copy.
# (Reversible: `uv run python snapshot.py restore` brings the broken baseline back.)
from helpers.dedup import find_duplicate_ids

before = client.count(COLLECTION).count
dup_ids = find_duplicate_ids(client)
client.delete(collection_name=COLLECTION, points_selector=dup_ids)
after = client.count(COLLECTION).count
print(f"deleted {len(dup_ids)} duplicates:  {before} -> {after} points")

In [ ]:
keys, rate = dup_rate("Tell me the Pokedex entry for Gengar")
print(f"duplicate rate: {rate:.0%}")   # now distinct documents fill the top-k
for k in keys: print(" ", k[0])

**Qdrant Web UI:** open `pokemon_viz` in the Visualize tab: the duplicate copies sit in tight clusters. Run the same dedup on it and refresh: the clusters thin out.

Dedup removes exact copies. What survives is near-duplicate *content*: ask about Gengar now and the top-5 is four different generations' flavor text for the same species — distinct documents, same information. That's a query-time diversity problem, and Qdrant handles it at search time with `group_by` (one result per document group) or MMR.

In [ ]:
# Same dedup on the visualization collection, then refresh the Web UI point cloud.
viz_dups = find_duplicate_ids(client, config.VIZ_COLLECTION)
client.delete(collection_name=config.VIZ_COLLECTION, points_selector=viz_dups)
print(f"pokemon_viz: removed {len(viz_dups)} duplicates")

## 3 · Fix #2: The Model That Was Fine at 151 Species Isn't Fine at 1,025
**Dashboard read:** duplicates are gone, but the paraphrase queries still miss — and the retrieval panel shows why: ask for "the electric mouse that stores electricity in its cheek pouches" and the 384d model returns Togedemaru, one of Pikachu's ten later-generation lookalikes. The model didn't get worse; the corpus grew until its vectors stopped separating neighbors.

Qdrant adds a candidate model to the live collection with zero downtime (named vectors, v1.18), so the experiment is safe: A/B both models on your own golden set, then commit — or roll back with the same one-line flip.

In [ ]:
# Pre-run offline. create_vector_name is instant; backfilling 23k points is the slow part
# and runs in the background. See prep.py. This is the zero-downtime migration:
#   client.create_vector_name(COLLECTION, config.DENSE_STRONG,
#       vector_name_config=models.DenseVectorNameConfig(
#           dense=models.DenseVectorConfig(size=1024, distance=models.Distance.COSINE)))
#   client.update_vectors(COLLECTION, points=[...])   # backfill, payload untouched
# Flip the agent to the candidate model in one line:
agent.set_retrieval(mode="bge")

rows = [json.loads(l) for l in Path("data/golden_dataset.jsonl").read_text().splitlines() if l.strip()]
fix2 = [r for r in rows if r["exercises"] == "fix2_embedding"]

# Instant gold-label check. Judged scores live on the Future AGI dashboard.
def recall(mode):
    hit = 0
    for r in fix2:
        top = {c["doc_id"] for c in agent.retrieve(r["query"], mode=mode)}
        hit += int(bool(top & set(r["gold_doc_ids"])))
    return hit / len(fix2)

print(f"recall@5  small (MiniLM 384d)={recall('minilm'):.2f}   large (bge-large 1024d)={recall('bge'):.2f}")

In [ ]:
# The eval confirms the win (0.64 -> 1.00 on the clone-crowded queries): keep bge.
# A regression would roll back with the same one-line flip — that's what the A/B protects.
print("retrieval mode:", agent.retrieval_state()["mode"])

The measurement is the lesson: **the model that was good enough on day one stopped being good enough when the corpus grew** — and only an A/B on your own golden set tells you that. The named-vector migration made the experiment safe: both models live on the same collection, the flip is one line, and it works in both directions — a regression rolls back the same way, zero downtime.

## 4 · Fix #3: Dense-Only → Hybrid Retrieval + Late-Interaction Reranking
**Dashboard read:** even with the strong model, the hardest paraphrase queries still miss. Groundedness stays green because the agent correctly says "I don't know," so only recall against the golden set exposes the failure.

The fix is one `query_points` call: **prefetch** dense + sparse candidates (sparse recovers exact words dense loses), **fuse** the two ranked lists with RRF, **rerank** the fused candidates with ColBERT token by token. The dense arm is bge — fix #3 builds on fix #2's migration, it doesn't replace it.

In [ ]:
q = "A Pokemon that lives in dark caves and uses sound waves to navigate and hunt."

# Dense-only — even the strong model — misses it: nothing in the query names Zubat,
# and the cave/echolocation wording is spread across many bat- and cave-dweller entries.
dense_only = [c["doc_id"] for c in agent.retrieve(q, mode="bge")]
print("dense-only   :", dense_only)

# The hybrid + rerank call. This is what mode='hybrid' runs. See agent.py.
sp = sparse([q], is_query=True)[0]
hybrid = client.query_points(
    COLLECTION,
    prefetch=[models.Prefetch(
        prefetch=[
            models.Prefetch(query=dense([q], config.MODEL_DENSE_STRONG, is_query=True)[0],
                            using=config.DENSE_STRONG, limit=20),
            models.Prefetch(query=models.SparseVector(indices=sp[0], values=sp[1]),
                            using=config.SPARSE, limit=20),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF), limit=20,
    )],
    query=colbert([q], is_query=True)[0], using=config.COLBERT, limit=5,
    with_payload=["doc_id"],
).points
print("hybrid+rerank:", [h.payload["doc_id"] for h in hybrid])

In [ ]:
# Flip the agent to hybrid, then score the queries dense retrieval was missing.
agent.set_retrieval(mode="hybrid")

import math
fix3 = [r for r in rows if r["exercises"] == "fix3_hybrid"]

# Instant gold-label check. Judged scores live on the Future AGI dashboard.
def score(mode):
    rec, ndcg, mrr = 0, 0, 0
    for r in fix3:
        seen, ranked = set(), []
        for c in agent.retrieve(r["query"], mode=mode):
            if c["doc_id"] not in seen: seen.add(c["doc_id"]); ranked.append(c["doc_id"])
        ranked, gold = ranked[:5], set(r["gold_doc_ids"])
        rec += int(any(d in gold for d in ranked))
        ndcg += sum(1/math.log2(i+2) for i,d in enumerate(ranked) if d in gold)  # idcg=1
        mrr += next((1/(i+1) for i,d in enumerate(ranked) if d in gold), 0)
    n = len(fix3); return rec/n, ndcg/n, mrr/n

print("recall / NDCG@5 / MRR   dense (bge): %.2f / %.2f / %.2f   hybrid: %.2f / %.2f / %.2f"
      % (*score("bge"), *score("hybrid")))

Two stages, measured separately — and neither carries it alone. The sparse arm widens the candidate pool (recall@20 0.89 → 0.94), but fusion *without* the rerank actually scores worse at top-5 (0.72) than pure dense (0.83): RRF mixes in keyword hits that push gold down. The ColBERT rerank is what turns the wider pool into wins: recall@5 0.83 → 1.00, NDCG@5 0.63 → 0.82. One Query API call covers prefetch, fusion, and rerank.

## 5 · Closing the Cold Open: One-Line Payload Filter
Dedup and hybrid didn't fix the opening failure: the stale pre-Gen-6 chart matches the question's wording, so it still wins retrieval. Better retrieval can't fix wrong data; metadata can. Every point carries `is_current`. Index it, filter on it.

In [ ]:
q = "Does the Steel type resist Ghost and Dark attacks?"
before = [c["doc_id"] for c in agent.retrieve(q, mode="hybrid") if c["name"] == "steel"]
print("no filter :", before)

client.create_payload_index(COLLECTION, "is_current", models.PayloadSchemaType.BOOL)
agent.set_retrieval(current_only=True)
after = [c["doc_id"] for c in agent.retrieve(q, mode="hybrid") if c["name"] == "steel"]
print("is_current:", after)

In [ ]:
# Ask the agent the cold-open question again. Now it is grounded in the current document.
answer, _ = agent.ask("Does the Steel type resist Ghost and Dark attacks?")
print(answer)

## 6 · Bonus: Multi-Hop In The Trace View (Future AGI)
A compound question makes the agent retrieve twice. Watch it split the task into two retriever spans in the Future AGI trace. That is the proof this is an agent, not a single-shot retriever.

In [ ]:
answer, chunks = agent.ask("What does Drowzee eat, and is that Pokemon weak to Bug-type attacks?")
print(answer)